# 06 · Swiss roll: ¿el espacio latente es el rectángulo?

Réplica de la figura *truth / d = 2 / 4 / 8 / …* con el latente 2-D de nuestros modelos. En cada panel están los puntos de **test**, coloreados por la coordenada verdadera s y colocados en una **lectura afín** de la representación sobre la carta (s, h):

- **DMAP**: lectura afín de ψ₁…ψ_m (el oráculo). Es exactamente lo que muestra la figura de referencia.
- **AE / VAE**: la media del latente μ(x), alineada con la carta mediante un mapa afín ajustado en **entrenamiento**. Un mapa afín no despliega nada, así que un pliegue o un colapso del latente siguen visibles.

Se muestra en cada fila la réplica **mediana** (por kNN/piso en la mayor m), no la mejor.

Dos métricas complementarias:
- **desplegado** = kNN(latente → carta) ≤ 10 × piso. Es invariante a reparametrizaciones suaves, así que un rectángulo *curvado* también cuenta como desplegado.
- **distorsión local** = Var log(|zᵢ − zⱼ| / |xᵢ − xⱼ|) sobre los 10 vecinos ambientales (0 = isometría salvo escala). Distingue un rectángulo de uno doblado o estirado.

| | Pregunta | Diseño | Criterio (fijado antes de correr) |
|---|---|---|---|
| **R1** | ¿AE y VAE obtienen el rectángulo por sí solos? | `ae`, `vae`, desde cero y por continuación, 6 réplicas | «rectángulo» = desplegado **y** distorsión ≤ 2 × la del oráculo DMAP en m = M |
| **R2** | ¿Un regularizador de isometría local lo consigue? | lo mismo con `reg = 'iso'` (vecinos ambientales) | tasa de rectángulo mayor que en R1 (Fisher, m = M) sin empeorar la mediana de MSE_x más de 2× |
| **R3** | ¿Un regularizador de isometría del decoder lo consigue? | lo mismo con `reg = 'jac'` (distorsión relajada de J_D) | igual que R2 |

Los dos regularizadores son invariantes a escala (no compiten con la KL) y actúan sobre μ(x). `iso` le pide al encoder que respete las distancias locales de x. `jac` le pide al decoder que no estire unas direcciones del latente más que otras. En el Swiss roll existe una carta isométrica (el rectángulo), así que ambos son compatibles con reconstrucción perfecta.

Duración estimada ~4.5 h en dos T4.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json', 'latent_figure.py']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina) y los regularizadores del latente. Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_06.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_06.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## Figura con resultados ya existentes (sin entrenar)

Si adjuntas como *Input* la salida del notebook 05 (`results_swiss_A_spectral`), esta celda dibuja DMAP, VAE desde cero y VAE por continuación con esas 16 réplicas. Si no está, la celda se omite y la misma figura sale de las corridas nuevas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from latent_figure import latent_grid

cand = [p.parent for p in sorted(Path('/kaggle/working').glob('results_swiss_A_spectral/metrics.csv'))] + \
       [p.parent for p in sorted(Path('/kaggle/input').rglob('results_swiss_A_spectral/metrics.csv'))]
if cand:
    F05 = cand[0]
    print('resultados del 05:', F05)
    fig, t05 = latent_grid([dict(label='DMAP (lectura afín)', folder=F05, model='oracle'),
                            dict(label='VAE desde cero', folder=F05, model='vae'),
                            dict(label='VAE continuación', folder=F05, model='vae_warm')],
                           [2, 4, 8, 16, 64], '/kaggle/working/latente_05',
                           'Swiss roll 60×10 (notebook 05): latente 2-D alineado afínmente a la carta (test)')
    plt.show()
else:
    print('No hay results_swiss_A_spectral: la figura saldrá de las corridas nuevas.')

## β

Factor 64: el elegido sin etiquetas para 60×10 (notebook 01a). Si ese barrido está en esta sesión, se lee de ahí.

In [ ]:
FACTOR = 64
for prev in [Path('/kaggle/working/beta_sweep_swiss/beta_selection.json')] + sorted(Path('/kaggle/input').rglob('beta_sweep_swiss/beta_selection.json')):
    if prev.exists():
        FACTOR = json.loads(prev.read_text())['selected_factor']
        break
print('factor beta =', FACTOR)

## Plan y estimación de tiempo

`REG_WEIGHT` es el peso λ del regularizador. Con λ = 1 el término pesa lo mismo que la reconstrucción cuando esta es del orden de 1 y domina al final del entrenamiento. Si quieres un barrido, añade entradas al `PLAN` con otros pesos (0.1, 10) y un `tag` distinto. El orden es la prioridad.

In [ ]:
import math
import protocol as P
BASE = json.loads((ROOT / 'config_swiss.json').read_text())
BASE.update(width=60., height=10.)
COMMON = dict(P.COMMON)
LADDER = [2, 4, 8, 16, 64]
M = max(LADDER)
SEEDS = list(range(6))
REG_WEIGHT = 1.0
comun = dict(beta_factor=FACTOR, seeds=SEEDS, modes=LADDER, models=['ae', 'vae'],
             m_protocols=['scratch', 'warm'], mode_order='spectral', oracle_decoder=False, reg_k=10)
PLAN = [
    dict(tag='R1_sin_reg', **comun),
    dict(tag='R2_iso', reg='iso', reg_weight=REG_WEIGHT, **comun),
    dict(tag='R3_jac', reg='jac', reg_weight=REG_WEIGHT, **comun),
]
for p in PLAN:
    p['out'] = f"/kaggle/working/results_swiss_latente_{p['tag']}"
SECONDS_PER_MODEL = 80
P.show_plan(PLAN, 'swiss', SECONDS_PER_MODEL, COMMON)
print('(jac hace 2 pasadas extra del decoder por lote: cuenta ~40 % más para R3)')

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(PLAN, BASE, ROOT, common=COMMON)

## Análisis

«Rectángulo» = desplegado (kNN ≤ 10 × piso) **y** distorsión local ≤ 2 × la distorsión del oráculo DMAP de la misma réplica en m = M. Las filas de inicio de cadena `warm` (= modelo desde cero en m = 2) se incluyen.

In [ ]:
from scipy.stats import fisher_exact
import protocol as P
R = {p['tag']: P.load(p['out']) for p in PLAN}

def con_rectangulo(d):
    q = P.with_chain_start(d).copy()
    ref = d[(d.model == 'oracle') & (d.m == M)].set_index('seed').local_distortion
    q['dist_ref'] = q.seed.map(ref)
    q['rectangulo'] = q.unfolded.astype(bool) & (q.local_distortion <= 2*q.dist_ref)
    return q

filas = []
for tag, d in R.items():
    if d is None:
        continue
    q = con_rectangulo(d)
    for (model, m), g in q.groupby(['model', 'm']):
        k, n = int(g.rectangulo.sum()), len(g)
        lo, hi = P.clopper_pearson(k, n)
        filas.append(dict(corrida=tag, model=model, m=int(m), desplegadas=f"{int(g.unfolded.astype(bool).sum())}/{n}",
                          rectangulo=f'{k}/{n}', tasa_rect=k/n, ic95_inf=lo, ic95_sup=hi,
                          distorsion_mediana=g.local_distortion.median(), knn_ratio_mediana=g.knn_chart_ratio.median(),
                          mse_x_mediana=g.mse_x.median()))
    o = d[d.model == 'oracle'].groupby('m').local_distortion.median()
    for m, v in o.items():
        filas.append(dict(corrida=tag, model='oracle (DMAP)', m=int(m), distorsion_mediana=v))
T = pd.DataFrame(filas)
display(T.style.format({'tasa_rect': '{:.0%}', 'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'distorsion_mediana': '{:.3g}',
                        'knn_ratio_mediana': '{:.3g}', 'mse_x_mediana': '{:.4g}'}, na_rep='')
        .set_caption('Rectángulo = desplegado y distorsión ≤ 2 × oráculo (m = M)'))

# ---- R1: sin regularizador ----
base = T[(T.corrida == 'R1_sin_reg') & (T.m == M) & T.model.isin(['ae', 'vae', 'ae_warm', 'vae_warm'])]
for _, r in base.iterrows():
    print(f"R1 {r.model:9s} m={M}: rectángulo {r.rectangulo} (desplegadas {r.desplegadas}), distorsión {r.distorsion_mediana:.3g}")

# ---- R2 / R3: con regularizador frente a sin él, en m = M ----
if R['R1_sin_reg'] is not None:
    q0 = con_rectangulo(R['R1_sin_reg'])
    for tag in ('R2_iso', 'R3_jac'):
        if R[tag] is None:
            continue
        q1 = con_rectangulo(R[tag])
        for model in ('ae', 'vae', 'ae_warm', 'vae_warm'):
            a = q1[(q1.model == model) & (q1.m == M)]
            b = q0[(q0.model == model) & (q0.m == M)]
            if not len(a) or not len(b):
                continue
            ka, kb = int(a.rectangulo.sum()), int(b.rectangulo.sum())
            p = fisher_exact([[ka, len(a)-ka], [kb, len(b)-kb]], alternative='greater')[1]
            costo = a.mse_x.median()/b.mse_x.median()
            ok = p < .05 and costo <= 2
            print(f"{tag} {model:9s}: rectángulo {ka}/{len(a)} vs {kb}/{len(b)} sin reg, Fisher p = {p:.3g}; "
                  f"MSE_x ×{costo:.2f} -> {'CUMPLE' if ok else 'no cumple'}")

In [ ]:
# ---- Figuras tipo 'truth / m = 2 / 4 / 8 / ...' para cada corrida ----
from latent_figure import latent_grid
for p in PLAN:
    if R[p['tag']] is None:
        continue
    rows = [dict(label='DMAP (lectura afín)', folder=p['out'], model='oracle')]
    rows += [dict(label=lab, folder=p['out'], model=mod) for lab, mod in
             (('AE desde cero', 'ae'), ('AE continuación', 'ae_warm'), ('VAE desde cero', 'vae'), ('VAE continuación', 'vae_warm'))]
    nombre = {'R1_sin_reg': 'sin regularizador', 'R2_iso': f"isometría local (λ = {p.get('reg_weight', 0):g})",
              'R3_jac': f"isometría del decoder (λ = {p.get('reg_weight', 0):g})"}[p['tag']]
    fig, _ = latent_grid(rows, LADDER, f"/kaggle/working/latente_{p['tag']}",
                         f"Swiss roll 60×10 · {nombre}: latente 2-D alineado afínmente a la carta (test)")
    plt.show()

# ---- Resumen: distorsión y tasa de rectángulo frente a m ----
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.4))
colores = {'R1_sin_reg': 'C7', 'R2_iso': 'C2', 'R3_jac': 'C1'}
estilo = {'ae': (':', '^'), 'ae_warm': ('-', '^'), 'vae': (':', 'o'), 'vae_warm': ('-', 'o')}
for i, (tag, d) in enumerate(R.items()):
    if d is None:
        continue
    q = con_rectangulo(d)
    for j, (model, (ls, mk)) in enumerate(estilo.items()):
        g = q[q.model == model]
        s = g.groupby('m').local_distortion.median()
        a.plot(s.index, s.values, color=colores[tag], ls=ls, marker=mk, label=f'{tag} · {model}')
        t = T[(T.corrida == tag) & (T.model == model)].sort_values('m')
        b.errorbar(t.m*1.03**(3*i+j-5), t.tasa_rect, yerr=[t.tasa_rect-t.ic95_inf, t.ic95_sup-t.tasa_rect],
                   color=colores[tag], ls=ls, marker=mk, capsize=2, label=f'{tag} · {model}')
    o = d[d.model == 'oracle'].groupby('m').local_distortion.median()
    a.plot(o.index, o.values, color='k', lw=1, alpha=.4 + .2*i)
a.plot([], [], color='k', lw=1, label='oráculo DMAP')
a.set(xscale='log', yscale='log', xlabel='m', ylabel='distorsión local (mediana)', title='Isometría del latente (0 = rectángulo)')
b.set(xscale='log', ylim=(-.05, 1.05), xlabel='m', ylabel='fracción «rectángulo»', title='Desplegado y no distorsionado (IC 95 %)')
for ax in (a, b):
    ax.set_xticks(LADDER); ax.set_xticklabels([str(m) for m in LADDER]); ax.minorticks_off()
b.legend(fontsize=6, ncol=2)
fig.tight_layout()
fig.savefig('/kaggle/working/latente_resumen.png', dpi=200); fig.savefig('/kaggle/working/latente_resumen.pdf')
plt.show()
T.to_csv('/kaggle/working/latente_tablas.csv', index=False)

## Qué descargar

`/kaggle/working/results_swiss_latente_*`, `latente_*.png/.pdf`, `latente_resumen.png/.pdf` y `latente_tablas.csv`.